In [2]:
import sys
import os
import numpy as np
import pandas as pd
import tensorflow as tf
import django
from PIL import Image as PilImage
import matplotlib.pyplot as plt

In [14]:
sys.path.append(os.path.abspath(".."))
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "food_advisor.settings")
django.setup()
from backend.models import Image

### Loading the model

In [33]:
model = tf.keras.models.load_model("models/base_model.keras")

In [83]:
class_names = ['apple_pie', 'baby_back_ribs',
 'baklava', 'beef_carpaccio', 'beef_tartare',
 'beet_salad',
 'beignets',
 'bibimbap',
 'pizza',
 'pork_chop',
 'poutine',
 'prime_rib',
 'ramen',
 'ravioli',
 'red_velvet_cake',
 'risotto',
 'samosa',
 'sashimi',
 'scallops','seaweed_salad']

label_dict = {}
label_dict = {}
for i, class_name in enumerate(class_names):
    label_dict[i] = class_name

label_dict

{0: 'apple_pie',
 1: 'baby_back_ribs',
 2: 'baklava',
 3: 'beef_carpaccio',
 4: 'beef_tartare',
 5: 'beet_salad',
 6: 'beignets',
 7: 'bibimbap',
 8: 'pizza',
 9: 'pork_chop',
 10: 'poutine',
 11: 'prime_rib',
 12: 'ramen',
 13: 'ravioli',
 14: 'red_velvet_cake',
 15: 'risotto',
 16: 'samosa',
 17: 'sashimi',
 18: 'scallops',
 19: 'seaweed_salad'}

In [81]:
def is_confident(prediction, threshold):
    if prediction.max() < threshold:
        return False
    
    return True

In [98]:
def make_prediction(image_path):
    image = PilImage.open(image_path)
    image = np.array(image.resize(size=(224, 224)))
    image = np.expand_dims(image, axis=0)
    prediction = model.predict(image)

    #fig, ax = plt.subplots(figsize=(12, 8))
    #labels = np.arange(1, 21)
    #ax.bar(labels, prediction.reshape(-1))
    #ax.xticks = [np.arange(0, 21)]
    print(is_confident(prediction, 0.7))

    options = [label_dict[option] for option in np.argsort(prediction.reshape(-1))[-3:][::-1]]
    prediction_info = {
        "prediction": label_dict[prediction.argmax()],
        "confidence": prediction.max(),
        "options": options
    }
    return prediction_info

make_prediction("../../FoodAdvisor/media/images/OIP_3.jpg")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
True


{'prediction': 'ravioli',
 'confidence': np.float32(0.7048729),
 'options': ['ravioli', 'pizza', 'beef_carpaccio']}

In [103]:
from backend.forms import ImageForm
from django.shortcuts import render, redirect

def home_page(request):
    is_food = False

    form = ImageForm()
    if request.method == "POST":
        form = ImageForm(request.POST, request.FILES)
        if form.is_valid():
            instance = form.save(commit=False)
            instance.user = request.user
            #instance.calories = calculate_calories()
            #instance.proteins = calculate_proteins()
            #instance.carbs = calculate_carbs()
            #instance.fats = calculate_fats()
            prediction_info = make_prediction(instance.image)
            print(type(instance.image))
            instance.save()
            
            is_food = True # To do: implement ML food recognition response
            if is_food:
                context = prediction_info
                return render(request, "backend/meal_summary.html", context)

    context = {"form": form}
    return render(request, "backend/home_page.html", context)


In [104]:
images = Image.objects.all()
images

SynchronousOnlyOperation: You cannot call this from an async context - use a thread or sync_to_async.

## Validation of the multiple layers 101 classes model

In [5]:
model = tf.keras.models.load_model("models/multiple_layers_model_v2.keras")
model.summary()

Model: "functional_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer_7 (InputLayer)      │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_2      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 101)            │       129,381 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,465,329 (13.22 MB)

 Trainable params: 538,981 (2.06 MB)

 Non-trainable params: 1,848,384 (7.05 MB)

 Optimizer params: 1,077,964 (4.11 MB)

In [ ]:
!pip install torch

In [ ]:
model.evaluate()